step 1:install the requirde packages

In [ ]:
!pip install -q accelerate peft bitsandbytes transformers trl

In [ ]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [ ]:
import torch

print(torch.cuda.device_count())
print(torch.cuda.get_device_name(0))

In [ ]:
import os
import torch
from datasets import load_dataset
from transformers import (
AutoModelForCausalLM,
AutoTokenizer,
BitsAndBytesConfig,
HfArgumentParser,
TrainingArguments,
pipeline,
logging
)

from peft import LoraConfig, PeftModel
from trl import SFTTrainer

## in the case of llama 2 the following prompt templates is used for the chat models

System prompt (optional) to guide the model
User prompt (required) to give the instruction
model Answer (required)


In [ ]:
# model that we get to train from hugging Face hub
model_name = "NousResearch/Llama-2-7b-chat-hf"

#the instruction dataset to use
dataset_name = "mlabonne/guanaco-llama2-1k"

# finetuned model name
new_model = "Llama-2-7b-chat-finetune"

#--------------------------
# Qlora parameter
#--------------------------

#Lora attention dimension
lora_r = 64 

# alpha parameter for lora scaling
lora_alpha=16

lora_dropout =0.1

#-------------------------
# bits and bytes parameter
#--------------------------

#activate 4-bit percisionbase model loading
use_4bit = True

#compute dtype for 4-bits base model
bnb_4bit_compute_dtype = "float16"

#Quantization type (fp4 or nf4)

bnb_4bit_quant_type = "nf4"

# Activated nested quantization fro 4-bit base models(double quantization)
use_nested_quant = False

#-----------------------------------------------
# Training Arguments parameters
#-----------------------------------------------

#Output directory where the model predictions and checkpoints will be stored 
output_dir= "./results"

#number of training epochs
num_train_epochs = 1

#Enable fp16/bf16 training(set bf16 to true with an a100)
fp16=False
bf16=False

#batch size per GPU for training 
per_device_train_batch_size = 1

#Batch size per GPU for evaluation
per_device_eval_batch_size = 4

#Number of updates steps to accumulate the gradients for 
gradient_accumulation_steps = 4

#Enable gradient checkpointing
gradient_ceckpointing = True

# Maximum fradient normal (gradient clipping)
max_grad_norm =0.3

#initial learning rate (Adamw optimizer)
learning_rate =2e-4

#Weight decay to apply to all layers except bias/LayerNorm weights
weight_decay = 0.001

#optimizer to use
optim = "paged_adamw_8bit"

#Learning rate schedule
lr_scheduler_type = "cosine"

#Number of training steps (overrides num_train_epochs)
max_steps = -1

# Ratio of steps for a linear warmup (from 0 to learning rate)
warmup_ratio = 100

# Group sequences into batches with same length
# Saves memory and speeds up training considerably
group_by_length =True

#Save checkpoints every x updates steps
save_steps = 0 

# Log every X updates steps
logging_steps = 25

##################################################
# SFT parameters 
#-----------------------------------------
# maximum sequence length to use
max_seq_length = None

# pack multiple short examples in the same input sequence to increase efficiency 
packing = False

#load the entire model on the GPU 0
device_map = {"":0}




# load evrithing and start fine tunning


In [ ]:
# load dataset 
dataset = load_dataset(dataset_name, split = "train")

#load tokenizer and model with QLora configuration
compute_dtype = getattr(torch, bnb_4bit_compute_dtype)

bnb_config = BitsAndBytesConfig(
    load_in_4bit = use_4bit,
    bnb_4bit_quant_type = bnb_4bit_quant_type,
    bnb_4bit_compute_dtype = compute_dtype,
    bnb_4bit_use_double_quant = use_nested_quant,
)

# check GPU capability
if compute_dtype == torch.float16 and use_4bit:
    major, _ = torch.cuda.get_device_capability()
    if major >= 8:
        print("=" * 80)
        print("Your GPU supports bfloat16: accelerate training  with bf16=True")
        print("=" * 80)

#load the base model
model =  AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config = bnb_config,
    device_map = device_map
)
model.config.use_cache = False
model.config.pretraining_tp = 1





In [ ]:
#load LLaMA tokenizer 
tokenizer  = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right" # fixed weird overflow issue with fp16 training

In [ ]:
#load LoRA configuration
peft_config = LoraConfig(
    lora_alpha=lora_alpha,
    lora_dropout=lora_dropout,
    r = lora_r,
    bias="none",
    task_type = "CAUSAL_LM"
)


In [ ]:

#Set training parameters
training_arguments = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs = num_train_epochs,
    per_device_train_batch_size = per_device_train_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    optim=optim,
    save_steps = save_steps,
    logging_steps=logging_steps,
    learning_rate = learning_rate,
    weight_decay=weight_decay,
    fp16=fp16,
    bf16=bf16,
    max_grad_norm=max_grad_norm,
    max_steps= max_steps,
    warmup_ratio = warmup_ratio,
    group_by_length=group_by_length,
    lr_scheduler_type = lr_scheduler_type,
    report_to = "tensorboard"
)







In [ ]:
# Set supervised fine-tuning parameters
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    args=training_arguments,
)

# Train model
trainer.train()

# save model


In [ ]:
trainer.save_model("/kaggle/working/Llama-2-7b-chat-finetune")

In [ ]:
trainer.model.save_pretrained("/kaggle/working/Llama-2-7b-chat-finetune")

In [ ]:
tokenizer.save_pretrained(
    "/kaggle/working/Llama-2-7b-chat-finetune"
)

In [ ]:
!find /kaggle/working/results -type f

In [ ]:
%tensorboard --logdir /kaggle/working/results/runs

In [ ]:
!kill 221

In [ ]:
%reload_ext tensorboard
%tensorboard --logdir /kaggle/working/results/runs

In [ ]:
%tensorboard --logdir /kaggle/working/results/runs

In [ ]:
logging.set_verbosity(logging.CRITICAL)

prompt = "what is large language model?"
pipe = pipeline(task="text-generation",model=model,tokenizer=tokenizer)
result = pipe(f"<s>[INST] {prompt} [/INST]")
print(result[0]['generated_text'])

# Save new llama mmodel


In [ ]:
!pip install -U "torchao>=0.17.0"

In [ ]:
from peft import PeftModel
new_model="/kaggle/working/Llama-2-7b-chat-finetune"
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    low_cpu_mem_usage=True,
    return_dict=True,
    torch_dtype=torch.float16,
    device_map="auto"
)

model = PeftModel.from_pretrained(base_model,new_model)
model =model.merge_and_unload()

tokenizer = AutoTokenizer.from_pretrained(model_name,trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

In [ ]:
merged_path = "/kaggle/working/merged_model"

model.save_pretrained(
    merged_path,
    safe_serialization=True
)

tokenizer.save_pretrained(merged_path)

print("Merged model saved to:", merged_path)

# Push model to hugging face

In [ ]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_Access_TOKEN")


In [ ]:
import locale
locale.getpreferredencoding = lambda:"UTF-8"

In [ ]:
!pip uninstall -y hf-xet huggingface-hub
!pip install -U huggingface-hub

In [ ]:
import huggingface_hub

print(huggingface_hub.__version__)

In [ ]:
import os

os.environ["HF_HUB_DISABLE_XET"] = "1"

In [ ]:
import os

os.environ["HF_HUB_DISABLE_XET"] = "1"

print("Xet disabled:", os.environ["HF_HUB_DISABLE_XET"])

In [ ]:
import os

os.environ["HF_Access_TOKEN"] = HF_TOKEN

In [ ]:
!hf auth login --token "$HF_Access_TOKEN"

In [ ]:
!hf auth whoami

In [ ]:
from huggingface_hub import HfApi

api = HfApi(token=HF_TOKEN)

print(api.whoami()["name"])

In [ ]:
repo_id = "Gunavant07/Llama-2-7b-chat-finetune"

api.create_repo(
    repo_id=repo_id,
    repo_type="model",
    exist_ok=True
)

print("Repository created/found successfully!")


In [ ]:
model.push_to_hub(
    repo_id,
    token=HF_TOKEN
)

tokenizer.push_to_hub(
    repo_id,
    token=HF_TOKEN
)

In [ ]:
merged_path = "/kaggle/working/merged_model"

model.save_pretrained(
    merged_path,
    safe_serialization=True
)

tokenizer.save_pretrained(merged_path)

print("Saved:", merged_path)

In [ ]:
!hf upload Gunavant07/Llama-2-7b-chat-finetune-1 /kaggle/working/merged_model . --repo-type model

In [ ]:
!hf upload Gunavant07/Llama-2-7b-chat-finetune-1 \
    /kaggle/working/merged_model \
    . \
    --repo-type model

In [ ]:
repo_id = "Gunavant07/Llama-2-7b-chat-finetune-1"

In [ ]:
import os
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"

In [ ]:
!HF_XET_HIGH_PERFORMANCE=1 hf upload \
    Gunavant07/Llama-2-7b-chat-finetune-1 \
    /kaggle/working/merged_model \
    . \
    --repo-type model \
    --token "$HF_TOKEN"

In [ ]:
!hf repos create {repo_id} --repo-type model --token "$HF_TOKEN"